### Graph Neural Network (GNN)

Drugs = Nodes in a graph
Known interactions = Edges between nodes
Features = Chemical properties (fingerprints, functional groups, physicochemical)
Task: Link prediction (will there be an edge between two nodes?)

## **Pipeline summary**

| Aspect | What you have |
|--------|----------------|
| **Data** | DrugBank DDI + SMILES; multi-view (Morgan, MACCS, phys-chem); stratified 70/15/15 train/val/test. |
| **Model** | Multi-view attention → GATv2 (2 layers) → edge classifier (binary + type); fits subgraph-based, memory-efficient training. |
| **Training** | Priority 1: Full-graph GNN once/epoch + edge-only batches (no k_hop per batch). AMP, graph on GPU, validate every N epochs, optional torch.compile. |
| **Metrics** | Validation: loss, accuracy, F1, precision, recall; test: same + ROC-AUC. |

The Pipeline in 5 Steps
```bash
1. FEATURE EXTRACTION
   Raw SMILES → Morgan FP + MACCS + Properties
   
2. GRAPH CONSTRUCTION
   Build adjacency matrix (who interacts with whom)
   
3. MULTI-VIEW FUSION
   Combine 3 views using attention
   
4. GNN ENCODING
   Message passing to learn node embeddings
   
5. EDGE CLASSIFICATION
   Predict interaction from node pair embeddings
```

### Metrices

- **Accuracy** (with a 0.90 target)
- **F1**
- **Precision**
- **Recall** (for “don’t miss DDIs”)
- **ROC-AUC**
to get a single, comparable test report for accuracy and F1/recall.

If you share your current test accuracy and F1/recall after one full run (with the Neg fix and new config), we can suggest next steps (e.g. one more GAT layer, dropout, or label smoothing) to push toward 90% while keeping the pipeline memory-efficient and fast.

In [19]:
import gc
import time
import json
import os
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Graph & Geometry
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import k_hop_subgraph
from torch.amp import GradScaler, autocast
from safetensors.torch import save_model, load_model

# Molecular Processing
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors, MACCSkeys

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, precision_recall_curve, auc, confusion_matrix

In [20]:
# Config
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_TYPE = DEVICE.type
TIMESTAMP = datetime.now().strftime("%d_%b_%H-%M")

# Paths
DDIS_DATA_PATH = 'dataset/drugdata/ddis.csv'
DRUG_SMILE_DATA_PATH = 'dataset/drugdata/drug_smiles.csv'

# Create Artifact Dirs
os.makedirs("models", exist_ok=True)
os.makedirs("images", exist_ok=True)

print(f"● System Ready. Device: {DEVICE}")


● System Ready. Device: cuda


- Small physical batch → Fits in memory
- Large effective batch → Better training stability
- This technique is used in GPT-3, BERT, and all large models!

In [ ]:

# OLD_CONFIG = {
#     # Memory Management
#     "physical_batch_size": 512,      # Increased (faster with cached embeddings)
#     "accumulation_steps": 4,         # Reduced (effective batch = 2048)
    
#     # Model Architecture
#     "hidden_dim": 256,               # Good balance
#     "n_heads": 4,                    # Increased for better attention
#     "dropout": 0.3,                  # Regularization
    
#     # Training Dynamics
#     "lr": 5e-4,                      # Lower for stability
#     "weight_decay": 1e-4,
#     "epochs": 80,                    # More epochs (but faster now!)
    
#     # Loss Configuration
#     "type_loss_weight": 0.5,
#     "use_class_weights": True,
#     "label_smoothing": 0.1,          # Prevent overconfidence
    
#     # Scheduler
#     "scheduler_patience": 5,
#     "scheduler_factor": 0.5,
#     "min_lr": 1e-6,
    
#     # System
#     "full_graph_on_gpu": True,
#     "validate_every_n_epochs": 2,
#     "use_compile": True,             # PyTorch 2.0+
    
#     # Data Augmentation
#     "edge_dropout_rate": 0.1,        # Drop 10% of edges during training
#     "feature_noise": 0.01,           # Add small noise to features
# }

# Cell: Enhanced Training Config for 90%+ Accuracy

CONFIG = {
    # Increased model capacity
    "physical_batch_size": 512,
    "accumulation_steps": 4,
    "hidden_dim": 384,           # Increased from 256
    "n_heads": 6,                # Increased from 4
    "dropout": 0.25,             # Slightly reduced
    "n_gat_layers": 3,           # Add 3rd GAT layer
    
    # Better training dynamics
    "lr": 3e-4,                  # Lower LR for fine-tuning
    "weight_decay": 5e-5,        # Reduced regularization
    "epochs": 100,               # More epochs
    
    # Loss configuration
    "type_loss_weight": 0.3,     # Reduced (focus on binary first)
    "use_class_weights": True,
    "focal_loss": True,          # Use focal loss for hard examples
    "focal_alpha": 0.7,          # Weight for positive class
    "focal_gamma": 2.0,          # Focus on hard examples
    
    # Scheduler
    "scheduler_patience": 7,     # More patience
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    
    # System
    "full_graph_on_gpu": True,
    "validate_every_n_epochs": 2,
    "use_compile": True,        # Disable if Triton issues
    
    # Data augmentation
    "edge_dropout_rate": 0.05,   # Reduced
    "feature_noise": 0.005,      # Reduced
}

print("✅ Enhanced config ready. To retrain:")
print("   model = ImprovedCardioMMADL(..., hidden_dim=384, n_heads=6)")
print("   model, history = train_cardio_model_optimized(graph, ENHANCED_TRAIN_CONFIG, model)")

✅ Enhanced config ready. To retrain:
   model = ImprovedCardioMMADL(..., hidden_dim=384, n_heads=6)
   model, history = train_cardio_model_optimized(graph, ENHANCED_TRAIN_CONFIG, model)


Why Multiview? Different views capture different aspects:
1. View 1 (Morgan FP): Substructure patterns. Good for structural similarity
2. View 2 (MACCS Keys): Functional groups. Good for functional groups (OH, COOH, etc.)
3. View 3 (Properties): Physical/chemical behavior. Good for metabolism.

### Deep Dive: Morgan Fingerprints
Imagine walking from each atom:

- Start at atom A
- Walk 2 steps in all directions
- Record what you see: "carbon-nitrogen-oxygen pattern"
- Hash this pattern → 1 bit in 1024-bit vector
- Repeat for all atoms

> Result: Binary vector capturing local substructures

`Why radius=2?`
- Radius 1: Too local (just immediate bonds)
- Radius 2: Good balance (captures small functional groups)
- Radius 3: Too broad (high memory, diminishing returns)

`Why fpSize=1024?`- 

- Smaller (512): More collisions (different patterns → same bit)
- 1024: Good balance
- Larger (2048): Sparse, high memory

`What are MACCS Keys?` Predefined 166 structural patterns (keys):
- Key 1: "Has O atom?"
- Key 50: "Has aromatic ring?"
- Key 120: "Has carboxylic acid group?"

In [22]:
class MultiViewFeatureExtractor:
    """
    Extracting diverse 'views' of the same molecule.
    View 1: Morgan FP (Detailed Substructure)
    View 2: MACCS Keys (Functional Groups)
    View 3: Physico-Chemical Properties (Global Properties)
    """
    def __init__(self):
        self.morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
        
    def get_features(self, smiles_df):
        view1_morgan = []
        view2_maccs = []
        view3_phys = []
        valid_ids = []
        
        print("● Extracting Multi-View Features...")
        
        for _, row in smiles_df.iterrows():
            mol = Chem.MolFromSmiles(str(row["smiles"]))
            if mol:
                # View 1: Morgan Fingerprint (1024 dim)
                fp = np.array(self.morgan_gen.GetFingerprint(mol), dtype=np.float32)
                
                # View 2: MACCS Keys (167 dim) - Great for functional groups
                maccs = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
                
                # View 3: Physico-chemical 8 properties (8 dim) - Great for transport/metabolism
                desc = np.array([
                    Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                    Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
                    Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
                    Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol)
                ], dtype=np.float32)
                
                view1_morgan.append(fp)
                view2_maccs.append(maccs)
                view3_phys.append(desc)
                valid_ids.append(row["drug_id"])
        
        # Stack and Normalize View 3 (since it has large values like Weight)
        v1 = np.stack(view1_morgan)
        v2 = np.stack(view2_maccs)
        v3 = np.stack(view3_phys)
        
        scaler = StandardScaler()
        v3 = scaler.fit_transform(v3)
        
        print(f"✓ Processed {len(valid_ids)} drugs.")
        print(f"  - View 1 Shape: {v1.shape}")
        print(f"  - View 2 Shape: {v2.shape}")
        print(f"  - View 3 Shape: {v3.shape}")
        
        return {
            "v1": torch.tensor(v1, dtype=torch.float),
            "v2": torch.tensor(v2, dtype=torch.float),
            "v3": torch.tensor(v3, dtype=torch.float),
            "ids": valid_ids
        }

# Load and Process
smiles_df = pd.read_csv(DRUG_SMILE_DATA_PATH)
extractor = MultiViewFeatureExtractor()
features_dict = extractor.get_features(smiles_df)

● Extracting Multi-View Features...
✓ Processed 1706 drugs.
  - View 1 Shape: (1706, 1024)
  - View 2 Shape: (1706, 167)
  - View 3 Shape: (1706, 8)


In [23]:
class DDIDataAugmenter:
    """
    Augment training data to improve generalization:
    1. Edge dropout (random edge masking)
    2. Feature perturbation
    3. Hard negative mining
    """
    
    @staticmethod
    def augment_graph_training(graph, edge_dropout=0.1, feature_noise=0.01):
        """
        Apply augmentation during training epoch.
        """
        # Edge dropout: Randomly remove edges from adjacency
        if edge_dropout > 0:
            mask = torch.rand(graph['edge_index'].shape[1]) > edge_dropout
            aug_edge_index = graph['edge_index'][:, mask]
        else:
            aug_edge_index = graph['edge_index']
        
        # Feature noise: Add small Gaussian noise
        if feature_noise > 0:
            aug_x_v1 = graph['x_v1'] + torch.randn_like(graph['x_v1']) * feature_noise
            aug_x_v2 = graph['x_v2'] + torch.randn_like(graph['x_v2']) * feature_noise
            aug_x_v3 = graph['x_v3'] + torch.randn_like(graph['x_v3']) * feature_noise
        else:
            aug_x_v1, aug_x_v2, aug_x_v3 = graph['x_v1'], graph['x_v2'], graph['x_v3']
        
        return {
            'edge_index': aug_edge_index,
            'x_v1': aug_x_v1,
            'x_v2': aug_x_v2,
            'x_v3': aug_x_v3
        }

### Graph Construction
Converting tabular data (CSV) → Graph structure (PyG format)


In [24]:
class MMADLGraphBuilder:
    def __init__(self, ddi_path, feature_dict):
        self.ddi_df = pd.read_csv(ddi_path)
        self.feats = feature_dict
        self.drug_map = {d: i for i, d in enumerate(feature_dict['ids'])}
        self.idx_map = {i: d for d, i in self.drug_map.items()}
        self.type_enc = LabelEncoder()
        
    def build(self):
        # Map Edges
        pos_src, pos_dst, pos_types = [], [], []
        neg_src, neg_dst = [], []
        
        print("● Building DDI Graph Topology...")
        for _, row in self.ddi_df.iterrows():
            if row['d1'] in self.drug_map and row['d2'] in self.drug_map:
                u, v = self.drug_map[row['d1']], self.drug_map[row['d2']]
                
                # Bidirectional Positive Edge
                # Because If A interacts with B, then B interacts with A
                pos_src.extend([u, v])
                pos_dst.extend([v, u])
                pos_types.extend([row['type'], row['type']])
                
                # Negative Edge Handling: "DRUGID$t" or "DRUGID$h" -> extract DRUGID only
                if isinstance(row['Neg samples'], str):
                    neg_raw = row['Neg samples'].strip().split('$')[0]
                    if neg_raw and neg_raw in self.drug_map:
                        w = self.drug_map[neg_raw]
                        neg_src.extend([u, w])
                        neg_dst.extend([w, u])

        # Convert to Tensors
        pos_edge_index = torch.tensor([pos_src, pos_dst], dtype=torch.long)
        neg_edge_index = torch.tensor([neg_src, neg_dst], dtype=torch.long)
        
        # Combine for processing
        full_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
        
        # Label Encoding: Encode Types
        # Original: ["metabolism", "toxicity", "bleeding"]
        # Encoded:  [48, 46, 19]
        y_types_enc = self.type_enc.fit_transform(pos_types)
        
        # Create Labels
        # Binary: 1 for pos, 0 for neg
        y_binary = torch.cat([torch.ones(pos_edge_index.shape[1]), torch.zeros(neg_edge_index.shape[1])]).long()
        # Type: Encoded for pos, -1 for neg
        y_type = torch.cat([torch.tensor(y_types_enc), torch.full((neg_edge_index.shape[1],), -1)]).long()
        
        return {
            "x_v1": self.feats['v1'],
            "x_v2": self.feats['v2'],
            "x_v3": self.feats['v3'],
            "edge_index": full_edge_index,
            "y_binary": y_binary,
            "y_type": y_type,
            "n_types": len(self.type_enc.classes_),
            "encoder": self.type_enc,
            "drug_map": self.drug_map
        }

    def create_stratified_split(self, graph):
        # Stratified Split based on Binary Labels to ensure Negatives are balanced in Train/Val
        print("● Creating Stratified Splits (TDCommons Standard)...")
        n_edges = graph['edge_index'].shape[1]
        indices = np.arange(n_edges)
        
        train_idx, temp_idx = train_test_split(
            indices, test_size=0.3, stratify=graph['y_binary'], random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, stratify=graph['y_binary'][temp_idx], random_state=42
        )
        
        # Create Boolean Masks
        for name, idx in zip(['train', 'val', 'test'], [train_idx, val_idx, test_idx]):
            mask = torch.zeros(n_edges, dtype=torch.bool)
            mask[idx] = True
            graph[f'{name}_mask'] = mask
            
        print(f"  - Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
        return graph

builder = MMADLGraphBuilder(DDIS_DATA_PATH, features_dict)
graph_data = builder.build()
graph = builder.create_stratified_split(graph_data)

● Building DDI Graph Topology...
● Creating Stratified Splits (TDCommons Standard)...
  - Train: 537062 | Val: 115085 | Test: 115085


Architecture Overview:
```
Input: 3 Views (Morgan, MACCS, Properties)
  ↓
[Projection Layers] → All views to same dimension
  ↓
[Attention Fusion] → Learn which view is most important
  ↓
[GAT Layer 1] → Message passing (2 heads)
  ↓
[GAT Layer 2] → Second round of message passing
  ↓
[Edge Classifier] → Predict interaction from node pairs
  ↓
Output: [Binary: interact?, Type: which type?]
```
### Edge Classifier: How edge prediction works:
```
Question: Do Drug A and Drug B interact?

Step 1: Get their embeddings
  emb_A = [0.2, 0.5, 0.1, ...]  (256-dim)
  emb_B = [0.8, 0.3, 0.9, ...]  (256-dim)

Step 2: Concatenate
  edge_feat = [emb_A, emb_B]  (512-dim)

Step 3: Pass through MLP
  hidden = MLP(edge_feat)  (256-dim)

Step 4: Predict
  binary = Linear(hidden)  (2-dim: [no_interact, interact])
  type = Linear(hidden)    (86-dim: which type?)
```

In [25]:
class FeatureAttentionLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.atten = nn.Linear(dim, 1)
        
    def forward(self, v1, v2, v3):
        stack = torch.stack([v1, v2, v3], dim=1)
        scores = F.softmax(self.atten(stack), dim=1)
        fused = torch.sum(stack * scores, dim=1)
        return fused, scores


class ImprovedCardioMMADL(nn.Module):
    """
    Improvements:
    1. Residual connections for better gradient flow
    2. Layer normalization for training stability
    3. Edge dropout for regularization
    4. Optional feature masking
    """
    def __init__(self, dims, hidden_dim, n_heads, n_types, dropout=0.3):
        super().__init__()
        
        # View Projectors with BatchNorm
        self.proj_v1 = nn.Sequential(
            nn.Linear(dims['v1'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v2 = nn.Sequential(
            nn.Linear(dims['v2'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v3 = nn.Sequential(
            nn.Linear(dims['v3'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # Feature Attention
        self.feat_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )
        
        # GATv2 Layers with residual connections
        self.gat1 = GATv2Conv(
            hidden_dim, hidden_dim // n_heads, 
            heads=n_heads, concat=True, dropout=dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        
        self.gat2 = GATv2Conv(
            hidden_dim, hidden_dim, 
            heads=1, concat=False, dropout=dropout
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        
        # Edge Classifier with deeper network
        self.edge_encoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )
        
        self.head_bin = nn.Linear(hidden_dim // 2, 2)
        self.head_type = nn.Linear(hidden_dim // 2, n_types)
        
        self.dropout = dropout

    def get_node_embeddings(self, x_v1, x_v2, x_v3, edge_index):
        """
        Full-graph forward with residual connections and normalization.
        """
        # Project each view
        h1 = self.proj_v1(x_v1)
        h2 = self.proj_v2(x_v2)
        h3 = self.proj_v3(x_v3)
        
        # Multi-view attention fusion
        stack = torch.stack([h1, h2, h3], dim=1)  # (N, 3, hidden_dim)
        scores = F.softmax(self.feat_attention(stack), dim=1)
        h_fused = torch.sum(stack * scores, dim=1)
        
        # GAT Layer 1 with residual
        h_gat1 = self.gat1(h_fused, edge_index)
        h_gat1 = self.norm1(h_gat1 + h_fused)  # Residual connection
        h_gat1 = F.elu(h_gat1)
        
        # GAT Layer 2 with residual
        h_gat2 = self.gat2(h_gat1, edge_index)
        node_emb = self.norm2(h_gat2 + h_fused)  # Skip connection to fusion
        
        return node_emb

        # : Remove edge dropout for now
    def forward_edges_from_emb(self, node_emb, edge_label_index):
        src, dst = edge_label_index[0], edge_label_index[1]
        # edge operator
        edge_feat = torch.cat([node_emb[src], node_emb[dst]], dim=-1)
        shared = self.edge_encoder(edge_feat)  # or self.classifier if using original model
        return self.head_bin(shared), self.head_type(shared)

    def forward(self, x_v1, x_v2, x_v3, edge_index, edge_label_index):
        """Full forward pass (for inference)."""
        node_emb = self.get_node_embeddings(x_v1, x_v2, x_v3, edge_index)
        return self.forward_edges_from_emb(node_emb, edge_label_index)  # Remove training=False

In [26]:
def train_cardio_model_optimized(graph, config, model, device='cuda'):
    """
    OPTIMIZED TRAINING: Compute node embeddings ONCE per epoch, cache, then train edges.
    Expected speedup: 10-20x (30min → 1.5-3min per epoch)
    """
    
    # Extract config
    physical_batch = config["physical_batch_size"]
    accum_steps = config["accumulation_steps"]
    type_w = config.get("type_loss_weight", 0.5)
    validate_every = config.get("validate_every_n_epochs", 2)
    
    print(f"🚀 OPTIMIZED TRAINING: Node embeddings cached once/epoch")
    print(f"   Batch={physical_batch}, accum={accum_steps}, effective={physical_batch * accum_steps}")
    
    # ============================================================
    # SETUP: Data Loaders
    # ============================================================
    train_edges = graph['edge_index'][:, graph['train_mask']]
    train_labels = torch.stack([
        graph['y_binary'][graph['train_mask']], 
        graph['y_type'][graph['train_mask']]
    ], dim=1)
    
    val_edges = graph['edge_index'][:, graph['val_mask']]
    val_labels = torch.stack([
        graph['y_binary'][graph['val_mask']], 
        graph['y_type'][graph['val_mask']]
    ], dim=1)
    
    # OPTIMIZATION: Use num_workers for parallel data loading
    train_loader = DataLoader(
        TensorDataset(train_edges.t(), train_labels),
        batch_size=physical_batch, 
        shuffle=True,
        pin_memory=True,
        num_workers=2,  # Parallel loading
        persistent_workers=True
    )
    
    val_loader = DataLoader(
        TensorDataset(val_edges.t(), val_labels),
        batch_size=physical_batch * 2,  # Larger for validation
        shuffle=False,
        pin_memory=True,
        num_workers=2
    )
    
    # ============================================================
    # SETUP: Loss Functions with Class Weights
    # ============================================================
    y_train_bin = graph['y_binary'][graph['train_mask']].numpy()
    n_pos, n_neg = int((y_train_bin == 1).sum()), int((y_train_bin == 0).sum())
    
    if config.get("use_class_weights", True) and n_pos > 0 and n_neg > 0:
        w_pos = (n_pos + n_neg) / (2.0 * n_pos)
        w_neg = (n_pos + n_neg) / (2.0 * n_neg)
        class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32, device=device)
        crit_bin = nn.CrossEntropyLoss(weight=class_weights)
        print(f"   Class weights (neg={w_neg:.3f}, pos={w_pos:.3f})")
    else:
        crit_bin = nn.CrossEntropyLoss()
    
    crit_type = nn.CrossEntropyLoss(ignore_index=-1)
    
    # ============================================================
    # SETUP: Move graph data to GPU once
    # ============================================================
    full_adj = graph['edge_index'].to(device)
    x_v1 = graph['x_v1'].to(device)
    x_v2 = graph['x_v2'].to(device)
    x_v3 = graph['x_v3'].to(device)
    print(f"   Graph + features on GPU ✓")
    
    # ============================================================
    # SETUP: Optimizer & Scheduler
    # ============================================================
    optimizer = Adam(
        model.parameters(), 
        lr=config['lr'], 
        weight_decay=config.get('weight_decay', 1e-4)
    )
    scaler = GradScaler()
    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max', 
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5),
        # verbose=True
    )
    
    # ============================================================
    # TRAINING LOOP
    # ============================================================
    history = {
        'train_loss': [], 'val_acc': [], 'val_f1': [], 
        'val_precision': [], 'val_recall': []
    }
    
    print(f"\n{'='*80}")
    print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Val Acc':<8} | {'Val F1':<8} | {'Val P':<8} | {'Val R':<8} | {'Time':<8}")
    print(f"{'='*80}")
    
    import time
    best_f1 = 0.0
    
    for epoch in range(1, config['epochs'] + 1):
        epoch_start = time.time()
        model.train()
        
        # ============================================================
        # KEY OPTIMIZATION: Compute node embeddings ONCE per epoch
        # ============================================================
        with torch.no_grad():
            with autocast(device.type):
                # Single full-graph forward pass
                node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                node_emb = node_emb.detach()  # Detach from computation graph
        
        # ============================================================
        # Edge Classification Training (Fast!)
        # ============================================================
        total_loss = 0.0
        optimizer.zero_grad()
        
        for i, (batch_edges, batch_labels) in enumerate(train_loader):
            batch_edges = batch_edges.t().to(device, non_blocking=True)
            y_bin = batch_labels[:, 0].to(device, non_blocking=True)
            y_type = batch_labels[:, 1].to(device, non_blocking=True)
            
            # Forward: Use cached node embeddings (no GNN computation!)
            # Mixed Precision Training (AMP)
            with autocast(device.type):
                pred_bin, pred_type = model.forward_edges_from_emb(node_emb, batch_edges)
                loss = (crit_bin(pred_bin, y_bin) + 
                       type_w * crit_type(pred_type, y_type)) / accum_steps
            
            # Backward with gradient accumulation
            scaler.scale(loss).backward()
            
            if (i + 1) % accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            total_loss += loss.item() * accum_steps
        
        train_loss = total_loss / len(train_loader)
        history['train_loss'].append(train_loss)
        
        # ============================================================
        # Validation
        # ============================================================
        do_validate = (epoch % validate_every == 0) or (epoch == config['epochs'])
        
        if do_validate:
            model.eval()
            all_preds, all_trues = [], []
            
            # Compute validation node embeddings once
            with torch.no_grad():
                with autocast(device.type):
                    node_emb_val = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                
                for batch_edges, batch_labels in val_loader:
                    batch_edges = batch_edges.t().to(device, non_blocking=True)
                    p_bin, _ = model.forward_edges_from_emb(node_emb_val, batch_edges)
                    all_preds.extend(torch.argmax(p_bin, dim=1).cpu().numpy())
                    all_trues.extend(batch_labels[:, 0].numpy())
            
            # Compute metrics
            all_trues = np.array(all_trues)
            all_preds = np.array(all_preds)
            
            val_acc = accuracy_score(all_trues, all_preds)
            val_f1 = f1_score(all_trues, all_preds, zero_division=0)
            val_prec = precision_score(all_trues, all_preds, zero_division=0)
            val_rec = recall_score(all_trues, all_preds, zero_division=0)
            
            history['val_acc'].append(val_acc)
            history['val_f1'].append(val_f1)
            history['val_precision'].append(val_prec)
            history['val_recall'].append(val_rec)
            
            scheduler.step(val_f1)
            
            # Early Stop: Save best model
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), f'models/best_model_f1_{val_f1:.4f}.pt')
            
            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | {val_acc:>6.4f}  | {val_f1:>6.4f}  | {val_prec:>6.4f}  | {val_rec:>6.4f}  | {epoch_time:>5.1f}s")
        else:
            # No validation this epoch
            for k in ['val_acc', 'val_f1', 'val_precision', 'val_recall']:
                history[k].append(history[k][-1] if history[k] else 0.0)
            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | (no val) | {epoch_time:>5.1f}s")
        
        torch.cuda.empty_cache()
    
    print(f"{'='*80}")
    print(f"✅ Training complete! Best Val F1: {best_f1:.4f}")
    
    return model, history

In [27]:
# Initialize improved model
model = ImprovedCardioMMADL(
    dims={'v1': 1024, 'v2': 167, 'v3': 8},
    hidden_dim=CONFIG['hidden_dim'],
    n_heads=CONFIG['n_heads'],
    n_types=graph['n_types'],
    dropout=CONFIG['dropout']
).to(DEVICE)

# Compile model for faster execution (PyTorch 2.0+)
if CONFIG['use_compile'] and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print("✅ torch.compile enabled")
    except:
        print("⚠️ torch.compile not available")

In [28]:
# Train with optimized function
model, history = train_cardio_model_optimized(
    graph, 
    CONFIG, 
    model, 
    device=DEVICE
)

🚀 OPTIMIZED TRAINING: Node embeddings cached once/epoch
   Batch=512, accum=4, effective=2048
   Class weights (neg=1.000, pos=1.000)
   Graph + features on GPU ✓

Epoch  | Train Loss  | Val Acc  | Val F1   | Val P    | Val R    | Time    
   1   |    1.3162   | (no val) |  81.7s
   2   |    1.0097   | 0.5000  | 0.6667  | 0.5000  | 1.0000  |  27.3s
   3   |    0.9117   | (no val) |   7.4s
   4   |    0.8628   | 0.5003  | 0.6667  | 0.5002  | 0.9997  |  11.0s
   5   |    0.8248   | (no val) |   6.8s
   6   |    0.8007   | 0.5117  | 0.6647  | 0.5061  | 0.9679  |  11.1s
   7   |    0.7739   | (no val) |   7.8s
   8   |    0.7552   | 0.5115  | 0.6540  | 0.5063  | 0.9232  |  13.5s
   9   |    0.7353   | (no val) |   7.7s
  10   |    0.7228   | 0.5151  | 0.6560  | 0.5083  | 0.9246  |  11.3s
  11   |    0.7107   | (no val) |   6.2s
  12   |    0.6974   | 0.5248  | 0.6108  | 0.5172  | 0.7458  |   9.8s
  13   |    0.6894   | (no val) |   5.5s
  14   |    0.6784   | 0.5729  | 0.6142  | 0.5601  | 

In [29]:
# Cell: Complete Test Set Evaluation
def comprehensive_test_evaluation(model, graph, batch_size=512):
    """
    Complete test set evaluation with:
    - Binary classification metrics
    - Type classification metrics
    - Confusion matrix
    - Per-class performance
    """
    print("\n" + "="*70)
    print("🎯 COMPREHENSIVE TEST SET EVALUATION")
    print("="*70)
    
    model.eval()
    test_edges = graph['edge_index'][:, graph['test_mask']]
    test_labels_bin = graph['y_binary'][graph['test_mask']]
    test_labels_type = graph['y_type'][graph['test_mask']]
    
    test_loader = DataLoader(
        TensorDataset(test_edges.t(), 
                     torch.stack([test_labels_bin, test_labels_type], dim=1)),
        batch_size=batch_size, shuffle=False
    )
    
    full_adj = graph['edge_index'].to(DEVICE)
    x_v1, x_v2, x_v3 = graph['x_v1'].to(DEVICE), graph['x_v2'].to(DEVICE), graph['x_v3'].to(DEVICE)
    
    all_preds_bin, all_probs_bin, all_trues_bin = [], [], []
    all_preds_type, all_trues_type = [], []
    
    with torch.no_grad():
        with autocast(DEVICE_TYPE):
            node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
        
        for batch_edges, batch_labels in test_loader:
            batch_edges = batch_edges.t().to(DEVICE)
            p_bin, p_type = model.forward_edges_from_emb(node_emb, batch_edges)
            
            # Binary predictions
            probs = torch.softmax(p_bin, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(p_bin, dim=1).cpu().numpy()
            all_preds_bin.extend(preds)
            all_probs_bin.extend(probs)
            all_trues_bin.extend(batch_labels[:, 0].numpy())
            
            # Type predictions (only for positive samples)
            type_preds = torch.argmax(p_type, dim=1).cpu().numpy()
            all_preds_type.extend(type_preds)
            all_trues_type.extend(batch_labels[:, 1].numpy())
    
    # Convert to arrays
    y_true_bin = np.array(all_trues_bin)
    y_pred_bin = np.array(all_preds_bin)
    y_prob_bin = np.array(all_probs_bin)
    y_true_type = np.array(all_trues_type)
    y_pred_type = np.array(all_preds_type)
    
    # ==================== BINARY CLASSIFICATION ====================
    print(f"\n{'─'*70}")
    print("📊 BINARY CLASSIFICATION (Interaction: Yes/No)")
    print(f"{'─'*70}")
    
    acc = accuracy_score(y_true_bin, y_pred_bin)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(y_true_bin, y_prob_bin)
    except:
        roc_auc = float('nan')
    
    print(f"  Accuracy:      {acc:.4f}  {'✅' if acc >= 0.90 else '⚠️  (Target: 0.90)'}")
    print(f"  F1 Score:      {f1:.4f}  {'✅' if f1 >= 0.85 else '⚠️  (Target: 0.85)'}")
    print(f"  Precision:     {prec:.4f}")
    print(f"  Recall:        {rec:.4f}  {'✅' if rec >= 0.85 else '⚠️  (Important: catch interactions!)'}")
    print(f"  ROC-AUC:       {roc_auc:.4f}")
    
    # Confusion Matrix

    cm = confusion_matrix(y_true_bin, y_pred_bin)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n  Confusion Matrix:")
    print(f"                    Predicted")
    print(f"                 No Int | Interact")
    print(f"    Actual No Int: {tn:6d} | {fp:6d}")
    print(f"    Actual Interact: {fn:6d} | {tp:6d}")
    print(f"\n  True Positives:  {tp:6d}  (Correctly detected interactions)")
    print(f"  False Positives: {fp:6d}  (False alarms - safe but flagged)")
    print(f"  False Negatives: {fn:6d}  (⚠️  CRITICAL: Missed interactions!)")
    print(f"  True Negatives:  {tn:6d}  (Correctly identified safe)")
    
    # ==================== TYPE CLASSIFICATION ====================
    print(f"\n{'─'*70}")
    print("🏷️  TYPE CLASSIFICATION (Which interaction type?)")
    print(f"{'─'*70}")
    
    # Filter to only positive samples (where type is meaningful)
    pos_mask = y_true_bin == 1
    if pos_mask.sum() > 0:
        y_true_type_pos = y_true_type[pos_mask]
        y_pred_type_pos = y_pred_type[pos_mask]
        
        type_acc = accuracy_score(y_true_type_pos, y_pred_type_pos)
        type_f1 = f1_score(y_true_type_pos, y_pred_type_pos, average='weighted', zero_division=0)
        
        print(f"  Type Accuracy: {type_acc:.4f}")
        print(f"  Type F1:       {type_f1:.4f}")
        print(f"  Total Types:   {len(np.unique(y_true_type_pos))} unique types in test set")
        
        # Top 5 most common types
        from collections import Counter
        type_counts = Counter(y_true_type_pos)
        print(f"\n  Top 5 Most Common Interaction Types:")
        for type_id, count in type_counts.most_common(5):
            type_name = graph['encoder'].inverse_transform([type_id])[0]
            accuracy = (y_pred_type_pos[y_true_type_pos == type_id] == type_id).mean()
            print(f"    Type {type_name}: {count:4d} samples, {accuracy:.2%} accuracy")
    
    print(f"\n{'='*70}")
    print(f"{'='*70}\n")
    
    return {
        'binary': {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec, 'roc_auc': roc_auc},
        'type': {'accuracy': type_acc, 'f1': type_f1} if pos_mask.sum() > 0 else {},
        'confusion_matrix': cm
    }

# Run comprehensive evaluation
test_results = comprehensive_test_evaluation(model, graph)


🎯 COMPREHENSIVE TEST SET EVALUATION

──────────────────────────────────────────────────────────────────────
📊 BINARY CLASSIFICATION (Interaction: Yes/No)
──────────────────────────────────────────────────────────────────────
  Accuracy:      0.8407  ⚠️  (Target: 0.90)
  F1 Score:      0.8539  ✅
  Precision:     0.7886
  Recall:        0.9311  ✅
  ROC-AUC:       0.8965

  Confusion Matrix:
                    Predicted
                 No Int | Interact
    Actual No Int:  43181 |  14362
    Actual Interact:   3966 |  53576

  True Positives:   53576  (Correctly detected interactions)
  False Positives:  14362  (False alarms - safe but flagged)
  False Negatives:   3966  (⚠️  CRITICAL: Missed interactions!)
  True Negatives:   43181  (Correctly identified safe)

──────────────────────────────────────────────────────────────────────
🏷️  TYPE CLASSIFICATION (Which interaction type?)
──────────────────────────────────────────────────────────────────────
  Type Accuracy: 0.8960
  Type F1: 

In [30]:
# Plot train vs val curves

In [ ]:
## Saving model (.safetensor)


In [31]:
# --- CLINICAL MAPPING ---
TYPE_MEANINGS = {
    48: "Metabolism Inhibition (High Risk)",
    46: "Toxicity Accumulation",
    69: "QT Prolongation (Arrhythmia Risk)",
    19: "Bleeding Risk Increase",
    "default": "Unspecified Mechanism"
}

In [36]:
# Cell: Enhanced Prediction Function

def predict_cardio_interaction_safe(drug_a_id, drug_b_id, model, graph, threshold=0.5):
    """
    Enhanced prediction with complete output:
    - Binary prediction (0 or 1)
    - Probability of interaction
    - Predicted interaction type (always shown)
    - Confidence level
    """
    model.eval()
    
    if drug_a_id not in graph['drug_map'] or drug_b_id not in graph['drug_map']:
        print("⚠️ Error: Drug IDs not found in database.")
        return None
        
    u = graph['drug_map'][drug_a_id]
    v = graph['drug_map'][drug_b_id]
    
    query_node_indices = torch.tensor([u, v], dtype=torch.long)
    
    # Extract 1-hop subgraph
    subset, edge_index, mapping, edge_mask = k_hop_subgraph(
        node_idx=query_node_indices, 
        num_hops=1, 
        edge_index=graph['edge_index'], 
        relabel_nodes=True
    )
    
    subgraph_u = mapping[0]
    subgraph_v = mapping[1]
    query_edge = torch.tensor([[subgraph_u], [subgraph_v]], device=DEVICE)
    
    # Get features
    x_v1 = graph['x_v1'][subset].to(DEVICE)
    x_v2 = graph['x_v2'][subset].to(DEVICE)
    x_v3 = graph['x_v3'][subset].to(DEVICE)
    edge_index = edge_index.to(DEVICE)
    
    with torch.no_grad():
        # Get predictions
        logits_bin, logits_type = model(x_v1, x_v2, x_v3, edge_index, query_edge)
        
        # Binary prediction
        probs_bin = torch.softmax(logits_bin, dim=1)[0]  # [prob_no_interaction, prob_interaction]
        prob_interaction = probs_bin[1].item()
        binary_pred = 1 if prob_interaction > threshold else 0
        
        # Type prediction (always compute, even if binary=0)
        probs_type = torch.softmax(logits_type, dim=1)[0]
        type_idx = torch.argmax(logits_type, dim=1).item()
        type_prob = probs_type[type_idx].item()
        
        # Get type name
        try:
            type_name = graph['encoder'].inverse_transform([type_idx])[0]
        except:
            type_name = type_idx
        
        # Confidence level
        confidence = "High" if max(prob_interaction, 1-prob_interaction) > 0.8 else \
                    "Medium" if max(prob_interaction, 1-prob_interaction) > 0.6 else "Low"
    
    # --- ENHANCED REPORT ---
    print(f"\n{'='*70}")
    print(f"💊 DRUG INTERACTION PREDICTION REPORT")
    print(f"{'='*70}")
    print(f"Drug A: {drug_a_id}")
    print(f"Drug B: {drug_b_id}")
    print(f"\n{'─'*70}")
    print(f"📊 BINARY CLASSIFICATION")
    print(f"{'─'*70}")
    print(f"  Prediction:          {'⚠️  INTERACTION DETECTED' if binary_pred == 1 else '✅ NO INTERACTION (Safe)'}")
    print(f"  Binary Output:       {binary_pred} ({'Interact' if binary_pred == 1 else 'Safe'})")
    print(f"  Interaction Prob:    {prob_interaction:.2%}")
    print(f"  Safe Prob:           {(1-prob_interaction):.2%}")
    print(f"  Confidence:          {confidence}")
    print(f"  Threshold:           {threshold:.2f}")
    
    print(f"\n{'─'*70}")
    print(f"🏷️  INTERACTION TYPE PREDICTION")
    print(f"{'─'*70}")
    print(f"  Predicted Type:      Type {type_name}")
    print(f"  Type Confidence:     {type_prob:.2%}")
    
    # Get type description if available
    type_desc = TYPE_MEANINGS.get(int(type_name) if str(type_name).isdigit() else type_name, 
                                   f"Interaction Type {type_name}")
    print(f"  Description:         {type_desc}")
    
    if binary_pred == 1:
        print(f"\n{'─'*70}")
        print(f"⚠️  CLINICAL RECOMMENDATION")
        print(f"{'─'*70}")
        if int(type_name) if str(type_name).isdigit() else -1 in [69, 19, 48, 46]:
            print(f"  🚨 HIGH RISK: This interaction requires immediate clinical review!")
            print(f"  Action: Consult physician before co-administration")
        else:
            print(f"  ⚠️  MODERATE RISK: Monitor for side effects")
            print(f"  Action: Discuss with healthcare provider")
    else:
        print(f"\n{'─'*70}")
        print(f"✅ ASSESSMENT: Combination appears safe based on model prediction")
        print(f"   Note: Model shows {type_prob:.1%} confidence in Type {type_name} IF interaction occurs")
        print(f"{'─'*70}")
    
    print(f"{'='*70}\n")
    
    # Return structured data
    return {
        'binary_prediction': binary_pred,
        'interaction_probability': prob_interaction,
        'predicted_type': int(type_name) if str(type_name).isdigit() else type_name,
        'type_probability': type_prob,
        'confidence': confidence,
        'drug_a': drug_a_id,
        'drug_b': drug_b_id
    }

# Test cases
print("="*70)
print("RUNNING TEST PREDICTIONS")
print("="*70)

# Test 1: Your original example
result1 = predict_cardio_interaction_safe('DB00404', 'DB01114', model, graph)

# Test 2: Try another pair (if available)
# Uncomment if you want to test more pairs:
# result2 = predict_cardio_interaction_safe('DB00181', 'DB00754', model, graph)

RUNNING TEST PREDICTIONS

💊 DRUG INTERACTION PREDICTION REPORT
Drug A: DB00404
Drug B: DB01114

──────────────────────────────────────────────────────────────────────
📊 BINARY CLASSIFICATION
──────────────────────────────────────────────────────────────────────
  Prediction:          ⚠️  INTERACTION DETECTED
  Binary Output:       1 (Interact)
  Interaction Prob:    92.78%
  Safe Prob:           7.22%
  Confidence:          High
  Threshold:           0.50

──────────────────────────────────────────────────────────────────────
🏷️  INTERACTION TYPE PREDICTION
──────────────────────────────────────────────────────────────────────
  Predicted Type:      Type 48
  Type Confidence:     99.73%
  Description:         Metabolism Inhibition (High Risk)

──────────────────────────────────────────────────────────────────────
⚠️  CLINICAL RECOMMENDATION
──────────────────────────────────────────────────────────────────────
  🚨 HIGH RISK: This interaction requires immediate clinical review!
  Actio